In [13]:
import pandas as pd
import numpy as np
from lxml import etree
import matplotlib.pyplot as plt 
import seaborn as sns

# Read paragraphs from XML

In [14]:
src_file = 'source.v1.tei.xml'
ns = {"tei": "http://www.tei-c.org/ns/1.0", 
      "xml": "http://www.w3.org/TR/html4"}
p_xpath = "//tei:body//tei:p"

In [15]:
tree = etree.parse(src_file)  # Replace with your actual file path
root = tree.getroot()
p_elements = root.xpath(p_xpath, namespaces=ns)

In [16]:
def extract_text_excluding_notes(elem):
    
    # Grab paragraph ID
    p_id_list = elem.xpath('./@xml:id', namespaces=ns)
    if len(p_id_list) == 0:
        p_id = None
    else:
        p_id = float(p_id_list[0].replace('p', ''))
    
    # Serialize the <p> element to string    
    p_str = etree.tostring(elem, encoding="unicode")
    
    # Wrap it in a dummy root element
    wrapped = f"<wrapper xmlns='http://www.tei-c.org/ns/1.0'>{p_str}</wrapper>"
    wrapper_elem = etree.fromstring(wrapped)

    # Remove all <note> elements
    for note in wrapper_elem.xpath(".//tei:note", namespaces=ns):
        note.getparent().remove(note)

    # Get all remaining <p> elements and concatenate their text
    text_parts = []
    for p in wrapper_elem.xpath(".//tei:p", namespaces=ns):
        text_parts.append(''.join(p.itertext()))
    p_text = ' '.join(text_parts)

    # Return id and text
    return (p_id, p_text)



In [17]:
paragraphs = [extract_text_excluding_notes(p) for p in p_elements]
paragraphs[:10]

[(None, 'POPOL WUJ'),
 (None, 'OJER TAQ TZIJ'),
 (None, "XB'AN PA TINAMIT K'ICHE',"),
 (None, "RAMAQ' K'ICHE' WINAQ"),
 (None, '\n        \n      '),
 (None, 'Versión crítica y actualizada'),
 (None, "Ajtz'ib'ab'"),
 (None, "Nab'e"),
 (1.0, "\n      \n      Are' uxe' ojer tzij waral K'iche' ub'i'."),
 (2.0, "\n      \n      Waral xchiqatz'ib'aj wi")]

# Creae DOC from paragraphs

In [18]:
# Convert to dataframe
DOC = pd.DataFrame(paragraphs, columns=['para_id', 'para_str'])
DOC.para_id = DOC.para_id.ffill()
DOC.loc[DOC.para_id.isna(), 'para_id'] = 0
DOC.para_id = DOC.para_id.astype(int)
DOC['line_id'] = DOC.groupby('para_id').cumcount() + 1
DOC = DOC.set_index(['para_id','line_id'])
DOC

para_str
para_id line_id                                    
0       1                                 POPOL WUJ
        2                             OJER TAQ TZIJ
        3                 XB'AN PA TINAMIT K'ICHE',
        4                      RAMAQ' K'ICHE' WINAQ
        5                        \n        \n      
...                                             ...
152     8              rumal maja b'i chi ilb'al re
        9              k'o nab'e ojer kumal ajawab'
        10                           sachinaq chik.
        11       Xere k'u ri mixutzinik chi konojel
        12                 K'iche' Sta. Cruz ub'i'.

[6278 rows x 1 columns]

# Create TOKEN from DOC

In [19]:
# Convert paragraphs to tokens
TOKEN = DOC.para_str.str.split(expand=True).stack().to_frame('token_str')
TOKEN.index.names = DOC.index.names + ['token_num']

In [20]:
# Create normalized terms from tokens
TOKEN['term_str'] = (
    TOKEN
        .token_str.str.replace(r"\d+", "", regex=True)
        .str.replace('.', '')
        .str.replace(',', '')
        .str.lower()
        .str.strip()
)

# Remove blanks
TOKEN = TOKEN[~TOKEN.term_str.str.match(r'^\s*$')]

In [25]:
TOKEN.head()

token_str term_str
para_id line_id token_num                   
0       1       0             POPOL    popol
                1               WUJ      wuj
        2       0              OJER     ojer
                1               TAQ      taq
                2              TZIJ     tzij

# Create DTM from TOKEN

In [ ]:
DTM = TOKEN.groupby(['para_id', 'term_str']).term_str.count().unstack(fill_value=0)
DTM

term_str,a,a',a'on,ab'aj,ab'ajil,ab'anel,ab'anoj,ab'i',ab'ix,ab'ixik,...,yan,ye'oltux,yitz',yitz'il,yojol,yolkwat,yoq'b'al,yoq'otajinaq,yub'jan,yujuj
para_id,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
149,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
150,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Extract VOCAB from DTM

In [22]:
VOCAB = DTM.sum().to_frame('n')
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()
VOCAB['i'] = np.log2(1/VOCAB.p)
VOCAB['df'] = DTM.astype(bool).sum()
VOCAB['dp'] = VOCAB.df / len(DTM)
VOCAB['di'] = np.log2(1/VOCAB.dp)
VOCAB['dh'] = VOCAB.dp * VOCAB.di
VOCAB

,n,p,i,df,dp,di,dh
term_str,,,,,,,
a,38,0.001447,9.432872,26,0.169935,2.556948,0.434514
a',6,0.000228,12.095837,6,0.039216,4.672425,0.183232
a'on,1,0.000038,14.680799,1,0.006536,7.257388,0.047434
ab'aj,26,0.000990,9.980359,20,0.130719,2.935460,0.383720
ab'ajil,1,0.000038,14.680799,1,0.006536,7.257388,0.047434
...,...,...,...,...,...,...,...
yolkwat,1,0.000038,14.680799,1,0.006536,7.257388,0.047434
yoq'b'al,1,0.000038,14.680799,1,0.006536,7.257388,0.047434
yoq'otajinaq,1,0.000038,14.680799,1,0.006536,7.257388,0.047434


# Create CHUNK from TOKEN

In [32]:
n_chunks = 60

TOKEN['chunk_num'] = pd.cut(TOKEN.reset_index().index, n_chunks, labels=[x for x in range(n_chunks)])
TOKEN

token_str term_str chunk_num
para_id line_id token_num                             
0       1       0             POPOL    popol         0
                1               WUJ      wuj         0
        2       0              OJER     ojer         0
                1               TAQ      taq         0
                2              TZIJ     tzij         0
...                             ...      ...       ...
152     11      5           konojel  konojel        59
        12      0           K'iche'  k'iche'        59
                1              Sta.      sta        59
                2              Cruz     cruz        59
                3            ub'i'.    ub'i'        59

[26264 rows x 3 columns]

In [49]:
CHUNK = TOKEN.groupby('chunk_num', observed=True).term_str.apply(lambda x: ' '.join(x)).to_frame('chunk_str')
# CHUNK['n_tokens'] = CHUNK.chunk_str.str.split().str.len()
CHUNK['n_tokens'] = TOKEN.chunk_num.value_counts().sort_index()
CHUNK

,chunk_str,n_tokens
chunk_num,,
0,popol wuj ojer taq tzij xb'an pa tinamit k'ich...,438
1,taj chijamataj chiwinaqir wa ulew ulaq'el ch'a...,438
2,nimachikop k'o chuwach ulew ta xraj k'u kitij ...,438
3,b'it katk'ix la uloq at uk'u'x kaj maqajisaj u...,437
4,ri kixk'ub' chitaninik chipe pa q'aq' tak'al c...,438
5,ri chi k'u ri kab'raqan chisilab' juyub' rumal...,438
6,kaqix ri mama' ati't kachb'ilan kib' a pa kixp...,438
7,at k'ajol k'o pa' achuch aqajaw maja b'i xcha'...,437
8,richilana mixqab'ano k'a xecha' chi kib'il kib...,438


# Save

In [46]:
TOKEN.to_csv("ajtzibab-TOKEN.csv", index=True)
VOCAB.to_csv("ajtzibab-VOCAB.csv", index=True)
CHUNK.to_csv(f"ajtzibab-CHUNK-{n_chunks}.csv", index=True)